# PulseAI — Customer Sentiment Analysis

### AI Major Capstone Project · Option 1: Customer Sentiment Analysis Dashboard

---

**Objective.** Build a system that automatically classifies customer feedback as
**negative / neutral / positive**, stores it, and surfaces sentiment trends and
issue drivers through an interactive dashboard.

This notebook is the analytical half of the project. It covers the modelling
work end to end and produces the artefacts the service consumes:

| Section | What it covers |
|---|---|
| 1 | Problem framing and the label space |
| 2 | Data acquisition and exploratory analysis |
| 3 | NLP preprocessing — two profiles, and why |
| 4 | Baseline: TF-IDF + Logistic Regression |
| 5 | DistilBERT fine-tuning (custom PyTorch loop) |
| 6 | Comparative evaluation and error analysis |
| 7 | Explainability |
| 8 | Latency and throughput |
| 9 | From predictions to business insight |
| 10 | Conclusions and limitations |

The production system built around this notebook lives in `api/` (FastAPI +
MongoDB Atlas) and `dashboard/` (React). See `README.md` to run it.

> **Running this notebook.** Training cells are guarded by a `RETRAIN` flag and
> default to loading the saved artefacts, so a full pass takes a couple of
> minutes. Set `RETRAIN = True` to re-run training from scratch (roughly 2 hours
> on CPU for the transformer).

In [ ]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Make the project package importable when the notebook runs from notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.config import LABELS, PATHS, TRAINING           # noqa: E402
from src.metrics import compute_metrics, format_metrics  # noqa: E402

RETRAIN = False          # True re-runs training instead of loading artefacts
RANDOM_STATE = TRAINING.seed

pd.set_option("display.max_colwidth", 110)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

# The same diverging palette the dashboard uses. Blue-for-positive rather than
# green is deliberate: red/green measures dE 4.1 under deuteranopia, which is
# indistinguishable for roughly 1 in 12 men. Red/blue measures 8.7.
SENTIMENT_COLORS = {"negative": "#e34948", "neutral": "#898781", "positive": "#2a78d6"}
PALETTE = [SENTIMENT_COLORS[label] for label in LABELS]

print(f"project root : {ROOT}")
print(f"label space  : {LABELS}")
print(f"python       : {sys.version.split()[0]}")

---
## 1. Problem framing

A mid-sized consumer business receives feedback through many channels at once —
app store reviews, support email, survey responses, social posts. Nobody reads
all of it. The questions that go unanswered are the operational ones:

- Is sentiment getting better or worse **this week**?
- Which channel is deteriorating?
- **What** are people actually complaining about, and how much of it is there?

### Why this is a 3-class problem

Binary positive/negative is the more common formulation and it is the wrong one
here. Most real feedback is *mixed* — "the food was good but we waited 40
minutes" — and forcing it to a pole either inflates the complaint rate or hides
it. A neutral class keeps that mass separate so the negative count means
something when someone escalates on it.

Neutral is also the hardest class, which is why **macro-F1 is the headline
metric** rather than accuracy. Accuracy lets a model that fails completely on
neutral still score well by getting the two easy classes right; macro-F1 does
not.

### Label space

The index order is part of the model contract — once a checkpoint is trained,
changing it silently mislabels every prediction.

In [ ]:
from src.config import ID2LABEL, LABEL2ID

print("index -> label:", ID2LABEL)
print("label -> index:", LABEL2ID)

---
## 2. Data acquisition and exploration

### Source and label mapping

The corpus is **Yelp Reviews** (`Yelp/yelp_review_full`): 650,000 real customer
reviews of businesses, labelled 1–5 stars. It is the closest public proxy for the
target domain — genuine customers writing about a service they paid for, at
realistic length, with realistic spelling.

Stars map to sentiment on the standard convention:

| Stars | Sentiment | Reasoning |
|---|---|---|
| 1–2 | negative | Unambiguous dissatisfaction |
| 3 | neutral | Mixed or indifferent — the "it was fine" band |
| 4–5 | positive | Unambiguous satisfaction |

This mapping is an assumption worth stating plainly: **3-star reviews are noisy
labels**. Some are genuinely balanced, some are politely negative. That noise
sets a practical ceiling on neutral-class performance, and Section 6 shows the
model hitting exactly that ceiling.

`src/dataset.py` streams the corpus and fills a per-class quota, so preparing a
balanced 16k sample never downloads the full multi-gigabyte archive.

In [ ]:
from src.dataset import load_splits, prepare

try:
    splits = load_splits()
except FileNotFoundError:
    print("Prepared splits not found — building them now (streams from the Hub)...")
    splits = prepare()

train_df, val_df, test_df = splits["train"], splits["val"], splits["test"]

summary = pd.DataFrame(
    {
        "rows": [len(train_df), len(val_df), len(test_df)],
        "negative": [int((df.label_name == "negative").sum()) for df in splits.values()],
        "neutral": [int((df.label_name == "neutral").sum()) for df in splits.values()],
        "positive": [int((df.label_name == "positive").sum()) for df in splits.values()],
    },
    index=["train", "val", "test"],
)
summary["balance"] = (summary[LABELS].min(axis=1) / summary[LABELS].max(axis=1)).round(3)
summary

The splits are **stratified**: every partition holds the class ratio of the whole.
Without that, a rare class can be absent from validation entirely, which silently
breaks macro-F1 — the metric averages over classes, and a class with no support
contributes a meaningless zero.

Deliberately sampling a balanced corpus is also a modelling decision. Real
feedback streams skew positive; training on that skew produces a model that
under-predicts the negative class, which is the one the business actually needs
detected. Balanced training plus `class_weight="balanced"` in the baseline
addresses that directly.

### Text length

Length drives two concrete choices: the transformer's `max_seq_length` (compute
cost scales with it) and whether truncation loses signal.

In [ ]:
train_df["n_words"] = train_df.text.str.split().str.len()

print(train_df.groupby("label_name")["n_words"].describe()[["mean", "50%", "75%", "max"]].round(1))

# Word count is only a proxy. What actually costs compute - and what decides
# whether a review gets truncated - is the *token* count after WordPiece
# splitting, and on real review text the two differ by a lot.
from transformers import AutoTokenizer  # noqa: E402

# Imported here rather than waiting for Section 3: the measurement below
# needs the light cleaning profile, and this cell is what motivates the
# preprocessing decisions that section then explains.
from src.preprocessing import clean_for_transformer  # noqa: E402

tokenizer_probe = AutoTokenizer.from_pretrained(TRAINING.base_checkpoint)
probe = train_df.text.head(800).map(clean_for_transformer)
token_lengths = np.array([
    len(tokenizer_probe(text, truncation=False)["input_ids"]) for text in probe
])
word_lengths = train_df.n_words.head(800).to_numpy()

print()
print(f"words  : mean {word_lengths.mean():.0f}   median {np.median(word_lengths):.0f}")
print(f"tokens : mean {token_lengths.mean():.0f}   median {np.median(token_lengths):.0f}"
      f"   p90 {np.percentile(token_lengths, 90):.0f}   max {token_lengths.max()}")
print(f"tokens per word: {token_lengths.mean() / word_lengths.mean():.2f}x")
print()
for cap in (128, 192, 256, 320, 512):
    print(f"  reviews fully inside {cap:>3} tokens: {(token_lengths <= cap).mean():6.1%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))

for label in LABELS:
    subset = train_df.loc[train_df.label_name == label, "n_words"]
    axes[0].hist(subset.clip(upper=300), bins=45, alpha=0.55,
                 label=label, color=SENTIMENT_COLORS[label])
axes[0].set(xlabel="words per review", ylabel="reviews", title="Review length by sentiment")
axes[0].legend(frameon=False, fontsize=9)

axes[1].hist(token_lengths.clip(max=600), bins=45, color="#2a78d6", alpha=0.85)
for cap, color in ((128, "#e34948"), (256, "#1baf7a")):
    coverage = (token_lengths <= cap).mean()
    axes[1].axvline(cap, color=color, lw=1.8, ls="--")
    axes[1].text(cap + 8, axes[1].get_ylim()[1] * 0.88,
                 f"{cap} tok\n{coverage:.0%} fit", fontsize=8.5, color=color)
axes[1].set(xlabel="WordPiece tokens per review", ylabel="reviews",
            title="Token length vs the truncation window")

plt.tight_layout()
plt.show()

**This measurement changed the project's design, so it is worth dwelling on.**

The intuitive move is to read the *word* distribution and pick a token budget from
it. That is exactly the mistake made on the first pass here. WordPiece splits rare
words, names and misspellings into several sub-tokens, so a review averaging ~110
words tokenises to ~160 tokens - roughly 1.4x - and the tail is far worse than the
mean suggests.

The consequence: at `max_seq_length = 128`, only **51%** of the corpus fits
(measured across all 16,002 prepared reviews, not a sample).
Every longer review is cut mid-text, and the model never sees the ending - which in
a review is often exactly where the verdict lands ("...but overall I would not go
back"). Meanwhile TF-IDF reads every word of every review, because a bag of words
has no length limit at all.

That asymmetry is a **handicap on the transformer, not a property of transformers**,
and Section 6 shows it costing real points. The corrected budget is
**`max_seq_length = 256`**, which covers **82%** of reviews. Section 6 reports both runs
side by side, because the difference between them is the clearest evidence in this
notebook that a preprocessing decision can matter more than the model.

Note also the length signal itself: negative reviews run longer than positive ones -
people explain complaints and are brief about praise. That is a property of *this*
corpus, not a universal truth, and it is one reason to be careful about transferring
this model to short formats such as tweets.

In [ ]:
# What does the raw text actually look like? Always look before cleaning.
for label in LABELS:
    example = train_df[train_df.label_name == label].iloc[0]
    print(f"--- {label.upper()} ({example.n_words} words) " + "-" * 40)
    print(repr(example.text[:320]))
    print()

Two data-quality issues are visible in the raw strings:

1. **Literal `\n` escape sequences.** The corpus stores newlines as the two
   characters backslash-and-n rather than a real newline. Left alone, that `n`
   fuses with the following word and the vocabulary fills with ghost tokens like
   `nthe`.
2. **Elongation and repeated punctuation** (`sooooo`, `!!!!!`) — real emphasis,
   but a vocabulary explosion if every repetition count is a distinct token.

Both are handled in the preprocessing pipeline below.

---
## 3. NLP preprocessing

Classical and neural models want **opposite** things from preprocessing, so
`src/preprocessing.py` exposes two profiles rather than one compromise.

### `clean_for_classical` — aggressive
TF-IDF has no notion of morphology or word order. `deliver`, `delivered` and
`delivery` are three unrelated dimensions unless you collapse them, so this
profile lowercases, expands contractions, strips punctuation and numbers,
removes stopwords and applies a conservative stemmer.

### `clean_for_transformer` — light
DistilBERT's WordPiece vocabulary already handles casing, punctuation and
sub-words, and its attention uses word order. Stripping punctuation here
destroys real signal — `"great!!!"` and `"great"` carry different intensity, and
`"NOT worth it"` depends on both the negation and the capitalisation.

### The one rule both profiles share: **never remove negations**

Dropping `not`, `no`, `never` is the classic sentiment-analysis bug. It turns
*"not good"* into *"good"* — a label flip introduced by the preprocessing itself.
The standard English stopword list contains all of them, so it is used here with
those words explicitly subtracted.

In [ ]:
from src.preprocessing import (
    STOPWORDS, KEYWORD_STOPWORDS, clean_for_classical, clean_for_transformer,
)

samples = [
    r"I don't think the DELIVERY was GOOOOOD!!!!! see https://x.com @acme #terrible service",
    r"Great spot.\nThe food was cold but the staff were lovely.",
    "This is not good at all.",
]

for text in samples:
    print("raw        :", text)
    print("transformer:", clean_for_transformer(text))
    print("classical  :", clean_for_classical(text))
    print()

print("Negations kept for the classifier:",
      [w for w in ("not", "no", "never", "very") if w not in STOPWORDS])

Note the last example: `"This is not good at all"` survives as `"not good"`, not
`"good"`. The negation is preserved and the word-bigram features in the baseline
can then represent `not_good` as a single discriminative feature.

### A third list, for a different job

Keyword extraction — the dashboard's "what are people talking about" panel — has
the **opposite** requirement to classification. `not` and `very` are essential
features for a model but they are not *topics*; a word cloud led by "not, but,
just" tells an analyst nothing. So `KEYWORD_STOPWORDS` keeps the negations and
adds conversational filler on top.

Same corpus, same tokeniser, two stopword lists, because they answer two
different questions.

In [ ]:
from src.preprocessing import extract_keywords

negative_text = train_df[train_df.label_name == "negative"].text.tolist()[:1500]
positive_text = train_df[train_df.label_name == "positive"].text.tolist()[:1500]

print("Topics in NEGATIVE feedback:")
print("  ", ", ".join(w for w, _ in extract_keywords(negative_text, top_n=16)))
print("\nTopics in POSITIVE feedback:")
print("  ", ", ".join(w for w, _ in extract_keywords(positive_text, top_n=16)))

In [ ]:
# Apply the cleaning profiles once and reuse them for the rest of the notebook.
started = time.perf_counter()

for frame in (train_df, val_df, test_df):
    frame["clean_classical"] = [clean_for_classical(t) for t in frame.text.astype(str)]
    frame["clean_transformer"] = [clean_for_transformer(t) for t in frame.text.astype(str)]

elapsed = time.perf_counter() - started
rows = len(train_df) + len(val_df) + len(test_df)
print(f"cleaned {rows:,} documents in {elapsed:.1f}s ({rows / elapsed:,.0f} docs/s)")

reduction = 1 - train_df.clean_classical.str.split().str.len().sum() / train_df.n_words.sum()
print(f"aggressive profile removes {reduction:.1%} of tokens")

---
## 4. Baseline — TF-IDF + Logistic Regression

**Why bother with a baseline at all?** Because *"we fine-tuned BERT and got 0.87
macro-F1"* is not a result — it is a number with no reference point. The baseline
answers the question a reviewer will actually ask: **how much did the transformer
buy us over a model that trains in thirty seconds?**

It has two other jobs:

1. **A data sanity check.** If a bag-of-words model scores near chance, the labels
   are wrong, not the model.
2. **A production fallback.** The API loads it automatically if the transformer
   checkpoint is missing, so the service always answers.

### Feature design

- **Word 1–2 grams** capture negation as a unit (`not_good` is its own feature).
- **Character 3–5 grams** absorb typos and morphology a stemmer misses —
  `dissapointed` and `disappointed` share most character n-grams but no word
  n-grams. Real customer feedback is full of typos.
- **`class_weight="balanced"`** counteracts residual class imbalance.
- **`sublinear_tf=True`** applies `1 + log(tf)`, so a review that says "bad" ten
  times is not ten times more negative than one that says it once.

In [ ]:
from src.train_baseline import build_pipeline, top_features_per_class

if RETRAIN:
    from src.train_baseline import train as train_baseline
    baseline_metrics = train_baseline(tune=True)
    pipeline = None
else:
    import joblib
    from src.metrics import load_model_metrics

    pipeline = joblib.load(PATHS.baseline_model)
    baseline_metrics = load_model_metrics()["models"]["baseline"]
    print(f"loaded baseline from {PATHS.baseline_model.name}")

print(format_metrics("BASELINE — TF-IDF + Logistic Regression (test set)", baseline_metrics))

### What did the model actually learn?

Logistic regression coefficients are directly readable, which makes this the
cheapest possible model audit. If the top features were dataset artefacts —
business names, formatting quirks — the model would be exploiting leakage rather
than learning sentiment.

In [ ]:
top_features = baseline_metrics.get("top_features")
if top_features is None and pipeline is not None:
    top_features = top_features_per_class(pipeline)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharex=False)
for ax, label in zip(axes, LABELS):
    features = top_features[label][:12][::-1]
    names = [name for name, _ in features]
    weights = [weight for _, weight in features]
    ax.barh(names, weights, color=SENTIMENT_COLORS[label], height=0.68)
    ax.set_title(f"{label} — strongest features", fontsize=10)
    ax.tick_params(labelsize=9)
    ax.grid(axis="y", visible=False)

plt.tight_layout()
plt.show()

These are sentiment words and phrases, not artefacts — the model is learning the
right thing. Notice the negated bigrams appearing among the negative features:
that is the preservation decision from Section 3 paying off directly.

---
## 5. DistilBERT fine-tuning

### Why DistilBERT rather than BERT-base

| | BERT-base | DistilBERT |
|---|---|---|
| Layers | 12 | 6 |
| Parameters | 110M | 66M |
| GLUE retention | 100% | ~97% |
| Relative inference cost | 1.0× | ~0.6× |

Losing ~3% of language understanding to halve the depth is the right trade for
this system. The API has to answer in real time, and the whole project was
trained and benchmarked **CPU-only** — BERT-base would roughly double an already
two-hour training run and push inference latency past the point where a live
dashboard demo feels responsive.

### Why a hand-written training loop

`src/train_transformer.py` implements the loop explicitly instead of calling
`transformers.Trainer`. For a capstone that is the point: the mechanics being
assessed are exactly what `Trainer` hides. The loop makes five decisions worth
naming:

1. **Decoupled weight decay.** Parameters are split into two groups: decay is
   applied to weight matrices but **not** to biases or LayerNorm parameters.
   Regularising LayerNorm measurably hurts fine-tuning — those parameters
   calibrate activation scale, and shrinking them toward zero fights the
   normalisation itself.

2. **Linear warmup then decay.** The classification head is randomly initialised
   while the encoder is pretrained. Full learning rate from step one sends large
   gradients from that random head back through the encoder and damages the
   pretrained representation — "catastrophic forgetting". Warmup over the first
   10% of steps lets the head find a reasonable region first.

3. **Gradient clipping at norm 1.0**, applied *after* unscaling. With mixed
   precision the gradients are scaled by a large constant; clipping before
   unscaling would compare that constant against the threshold and do nothing.

4. **Dynamic padding.** Sequences are padded per batch to the batch's own longest
   member, not to a fixed 128. Padding everything to the maximum makes every
   batch cost what the longest review in the corpus costs. Attention masks
   already tell the model to ignore pad positions, so this is a free throughput
   win.

5. **Checkpoint on validation macro-F1, never on training loss.** Training loss
   falls monotonically whether or not the model is generalising. Selecting on it
   reliably picks the most overfitted epoch.

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from src.train_transformer import DynamicPadCollator, SentimentDataset, resolve_device

device = resolve_device("auto")
print(f"device: {device} · torch {torch.__version__}")

tokenizer = AutoTokenizer.from_pretrained(TRAINING.base_checkpoint)

# Demonstrate what dynamic padding actually saves on this corpus.
sample = [clean_for_transformer(t) for t in train_df.text.head(512)]
lengths = np.array([len(tokenizer(t, truncation=True, max_length=128)["input_ids"])
                    for t in sample])

fixed_cost = len(lengths) * 128
batched = lengths.reshape(-1, 16)
dynamic_cost = (batched.max(axis=1) * 16).sum()

print(f"\ntoken length  mean {lengths.mean():.0f} · median {np.median(lengths):.0f} "
      f"· 95th {np.percentile(lengths, 95):.0f}")
print(f"padded to max_length : {fixed_cost:,} token slots")
print(f"padded per batch     : {dynamic_cost:,} token slots "
      f"({1 - dynamic_cost / fixed_cost:.1%} saved)")

In [ ]:
from src.metrics import load_model_metrics

if RETRAIN:
    from src.train_transformer import train as train_transformer
    train_transformer(max_length=256, epochs=3)          # hours on CPU

all_models = load_model_metrics()["models"]

# Preference order, so the notebook still runs if only one transformer run
# exists: the 256-token run is the final model, the 128-token run is the
# ablation kept as its control.
for key in ("distilbert", "distilbert_seq128"):
    if key in all_models:
        bert_metrics = all_models[key]
        break
else:
    raise RuntimeError(
        "No transformer results in reports/metrics.json. "
        "Run: python -m src.train_transformer --max-length 256 --epochs 3"
    )

print(f"loaded results for: {bert_metrics['model']}")
print(format_metrics("DISTILBERT FINE-TUNED (test set)", bert_metrics))

In [ ]:
history = pd.DataFrame(bert_metrics.get("history") or [])

if history.empty:
    # A run interrupted before it finished writing training_history.json still
    # produces a usable checkpoint; it just has no curve to plot.
    print("No per-epoch history recorded for this checkpoint.")
    print("Re-run training, or recover it from reports/logs/*.log.")
else:
    display(history)

    fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.6))

    axes[0].plot(history.epoch, history.train_loss, marker="o",
                 color="#2a78d6", lw=2, label="train")
    axes[0].plot(history.epoch, history.val_loss, marker="o",
                 color="#eb6834", lw=2, label="validation")
    axes[0].set(xlabel="epoch", ylabel="cross-entropy loss", title="Loss")
    axes[0].set_xticks(history.epoch)
    axes[0].legend(frameon=False, fontsize=9)

    axes[1].plot(history.epoch, history.val_f1_macro, marker="o",
                 color="#2a78d6", lw=2, label="macro F1")
    axes[1].plot(history.epoch, history.val_accuracy, marker="o",
                 color="#1baf7a", lw=2, label="accuracy")
    axes[1].set(xlabel="epoch", ylabel="score", title="Validation score", ylim=(0, 1))
    axes[1].set_xticks(history.epoch)
    axes[1].legend(frameon=False, fontsize=9)

    plt.tight_layout()
    plt.show()

    print(f"best epoch by validation macro-F1: {bert_metrics['best_epoch']} "
          f"({bert_metrics['val_f1_macro']})")
    if bert_metrics.get("training_note"):
        print()
        print(bert_metrics["training_note"])

The gap between the training and validation loss curves is the overfitting
signal. Training loss falling while validation loss flattens or rises means the
model has started memorising rather than generalising — and it is precisely why
the checkpoint is selected on validation macro-F1 instead.

---
## 6. Comparative evaluation

Both models are now evaluated on the **same held-out test set**, which neither
model saw during training and which was not used for checkpoint selection
(validation did that job). This is the only comparison that means anything.

In [ ]:
comparison = pd.DataFrame(
    {
        "TF-IDF + LogReg": [baseline_metrics[k] for k in ("accuracy", "f1_macro", "f1_weighted")],
        "DistilBERT": [bert_metrics[k] for k in ("accuracy", "f1_macro", "f1_weighted")],
    },
    index=["Accuracy", "F1 (macro)", "F1 (weighted)"],
).round(4)
comparison["Δ"] = (comparison["DistilBERT"] - comparison["TF-IDF + LogReg"]).round(4)
comparison["Δ %"] = (comparison["Δ"] / comparison["TF-IDF + LogReg"] * 100).round(1)
comparison

### The truncation ablation

The first fine-tuning run used `max_seq_length = 128` and **lost to the TF-IDF
baseline**. That result is kept here rather than deleted, because the diagnosis is
the most useful thing in this notebook.

The cause was not the model. It was that 128 tokens covers only 51% of the corpus,
so the transformer was reading half of every long review while the baseline read all
of it. Re-running at 256 tokens - same architecture, same data, same
hyper-parameters, one changed number - is the controlled experiment that isolates it.

In [ ]:
ablation = pd.DataFrame([
    {
        "run": all_models[key]["model"],
        "max_seq_length": all_models[key].get("hyperparameters", {}).get("max_seq_length", "n/a"),
        "accuracy": all_models[key]["accuracy"],
        "f1_macro": all_models[key]["f1_macro"],
        "neutral_f1": all_models[key]["per_class"]["neutral"]["f1"],
        "latency_ms": all_models[key].get("latency_ms_per_sample"),
    }
    for key in ("baseline", "distilbert_seq128", "distilbert", "distilbert_int8")
    if key in all_models
]).round(4)
display(ablation)

if {"distilbert", "distilbert_seq128"} <= set(all_models):
    gain = all_models["distilbert"]["f1_macro"] - all_models["distilbert_seq128"]["f1_macro"]
    print()
    print(f"Context window 128 -> 256 is worth {gain * 100:+.1f} points of macro-F1.")
    print("Same model, same data, same hyper-parameters - one changed number.")

if "distilbert_int8" in all_models:
    int8 = all_models["distilbert_int8"]
    fp32 = all_models["distilbert"]
    print()
    print("Quantization (INT8, the build actually deployed):")
    print(f"  macro-F1  {fp32['f1_macro']:.4f} -> {int8['f1_macro']:.4f}"
          f"  ({int8['f1_macro'] - fp32['f1_macro']:+.4f})")
    print(f"  size      {int8.get('size_mb_original', 0):.0f} MB -> {int8.get('size_mb', 0):.0f} MB")
    print(f"  latency   {fp32.get('latency_ms_per_sample')} ms -> "
          f"{int8.get('latency_ms_per_sample')} ms")
    print("  Quantizing cost nothing measurable in accuracy and made a 512 MB")
    print("  free container viable - see scripts/quantize_model.py.")

In [ ]:
per_class = pd.DataFrame(
    [
        {
            "class": label,
            "baseline F1": baseline_metrics["per_class"][label]["f1"],
            "DistilBERT F1": bert_metrics["per_class"][label]["f1"],
            "DistilBERT precision": bert_metrics["per_class"][label]["precision"],
            "DistilBERT recall": bert_metrics["per_class"][label]["recall"],
            "support": bert_metrics["per_class"][label]["support"],
        }
        for label in LABELS
    ]
).round(3)
per_class["gain"] = (per_class["DistilBERT F1"] - per_class["baseline F1"]).round(3)
display(per_class)

fig, ax = plt.subplots(figsize=(7.5, 3.4))
x = np.arange(len(LABELS))
ax.bar(x - 0.19, per_class["baseline F1"], 0.36, label="TF-IDF + LogReg", color="#2a78d6")
ax.bar(x + 0.19, per_class["DistilBERT F1"], 0.36, label="DistilBERT", color="#eb6834")
for i, (base, bert) in enumerate(zip(per_class["baseline F1"], per_class["DistilBERT F1"])):
    ax.text(i - 0.19, base + 0.015, f"{base:.3f}", ha="center", fontsize=8.5)
    ax.text(i + 0.19, bert + 0.015, f"{bert:.3f}", ha="center", fontsize=8.5)
ax.set(xticks=x, xticklabels=LABELS, ylabel="F1", ylim=(0, 1.05),
       title="Per-class F1 on the held-out test set")
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))

for ax, (name, metrics) in zip(
    axes, [("TF-IDF + LogReg", baseline_metrics), ("DistilBERT", bert_metrics)]
):
    matrix = np.array(metrics["confusion_matrix"])
    normalised = matrix / matrix.sum(axis=1, keepdims=True)
    annotations = np.array([[f"{c}\n{p:.0%}" for c, p in zip(rc, rp)]
                            for rc, rp in zip(matrix, normalised)])
    sns.heatmap(normalised, annot=annotations, fmt="", cmap="Blues", vmin=0, vmax=1,
                xticklabels=LABELS, yticklabels=LABELS, cbar=False, ax=ax,
                linewidths=2, linecolor="white", annot_kws={"fontsize": 10})
    ax.set(title=f"{name} — macro F1 {metrics['f1_macro']:.3f}",
           xlabel="predicted", ylabel="true")

plt.tight_layout()
plt.show()

### Reading the errors

The structure of the mistakes matters more than their count. Confusions fall into
two very different categories:

- **Adjacent errors** (negative↔neutral, neutral↔positive). Low cost. The
  feedback still lands in roughly the right place for triage, and a human
  reviewing the neutral bucket will catch it.
- **Polar errors** (negative↔positive). Genuinely expensive — an angry customer
  filed as happy is a complaint that never gets actioned.

A model with the same accuracy but its errors concentrated in the polar corners
would be far worse for this use case, which is why the raw accuracy number is not
the thing to optimise.

In [ ]:
matrix = np.array(bert_metrics["confusion_matrix"])
total = matrix.sum()
correct = np.trace(matrix)
polar = matrix[0, 2] + matrix[2, 0]
adjacent = total - correct - polar

print(f"correct        : {correct:5d}  ({correct / total:.1%})")
print(f"adjacent errors: {adjacent:5d}  ({adjacent / total:.1%})  low cost")
print(f"polar errors   : {polar:5d}  ({polar / total:.1%})  expensive")
print(f"\nOf all errors, {polar / (total - correct):.1%} are polar.")

### Confidence as a triage signal

A well-calibrated model should be *less* confident when it is wrong. If that
holds, confidence becomes an operational tool: route low-confidence predictions
to a human and let the rest through automatically.

In [ ]:
confidence_rows = []
for name, metrics in [("baseline", baseline_metrics), ("DistilBERT", bert_metrics)]:
    if "mean_confidence" in metrics:
        confidence_rows.append({
            "model": name,
            "overall": round(metrics["mean_confidence"], 3),
            "when correct": round(metrics["mean_confidence_correct"], 3),
            "when wrong": round(metrics["mean_confidence_incorrect"], 3),
            "gap": round(metrics["mean_confidence_correct"] - metrics["mean_confidence_incorrect"], 3),
        })

display(pd.DataFrame(confidence_rows).set_index("model"))
print("A positive gap means low confidence is a usable signal for human review.")

---
## 7. Explainability

A dashboard that tells an operations team *"this is negative, 94% confident"* and
nothing else will not be trusted, and should not be. The API therefore returns a
per-word attribution alongside every prediction.

**Method: leave-one-out occlusion.** Each word is deleted in turn and the drop in
the predicted class probability is that word's contribution. It costs one forward
pass per word, which is why it is opt-in — but it has three properties that suit a
business tool:

- **Model-agnostic.** Works identically for the transformer, the baseline and the
  lexicon fallback.
- **No gradients required.** Nothing to configure, nothing to go wrong at serving
  time.
- **Explainable to a non-ML stakeholder** in one sentence: *"we removed this word
  and the score moved this much."*

In [ ]:
from api.inference import engine

engine.load()
print(f"serving backend: {engine.backend} · {engine.model_name}\n")

demo = "The delivery was three days late and the box arrived damaged, but support refunded me quickly."
prediction = engine.predict(demo, explain=True)

print(f"prediction : {prediction.label}  ({prediction.confidence:.1%} confident)")
print(f"scores     : { {k: round(v, 3) for k, v in prediction.scores.items()} }")
print(f"categories : {prediction.categories}\n")

attribution = (
    pd.DataFrame(prediction.explanation)
    .sort_values("weight", ascending=False)
    .head(10)
    .reset_index(drop=True)
)
display(attribution)

In [ ]:
tokens = pd.DataFrame(prediction.explanation)

fig, ax = plt.subplots(figsize=(9, max(2.6, len(tokens) * 0.26)))
colors = ["#e34948" if w > 0 else "#898781" for w in tokens.weight]
ax.barh(range(len(tokens)), tokens.weight, color=colors, height=0.7)
ax.set_yticks(range(len(tokens)))
ax.set_yticklabels(tokens.token, fontsize=9)
ax.invert_yaxis()
ax.axvline(0, color="#0b0b0b", lw=1)
ax.set(xlabel=f"contribution to '{prediction.label}'",
       title="Leave-one-out word attribution")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

---
## 8. Latency and throughput

The requirement is *real-time API inference*, so latency is a result, not an
afterthought. Two numbers matter and they are different:

- **Single-sample latency** — what one user waits for on the Analyze page.
- **Batched throughput** — what a bulk CSV import achieves, where one padded
  forward pass covers many documents.

In [ ]:
texts = test_df.text.head(64).tolist()

# Warm up: the first forward pass pays lazy kernel and allocator costs.
engine.predict_batch(texts[:4])

single_times = []
for text in texts[:24]:
    started = time.perf_counter()
    engine.predict(text)
    single_times.append((time.perf_counter() - started) * 1000)

rows = []
for size in (1, 8, 32, 64):
    batch = texts[:size]
    started = time.perf_counter()
    engine.predict_batch(batch)
    elapsed = time.perf_counter() - started
    rows.append({
        "batch size": size,
        "total ms": round(elapsed * 1000, 1),
        "ms / sample": round(elapsed / size * 1000, 2),
        "docs / sec": round(size / elapsed, 1),
    })

print(f"single-sample latency  p50 {np.percentile(single_times, 50):.1f} ms · "
      f"p95 {np.percentile(single_times, 95):.1f} ms")
print(f"reported at training   {bert_metrics.get('latency_ms_per_sample', 'n/a')} ms\n")
display(pd.DataFrame(rows).set_index("batch size"))

Throughput per document improves substantially with batch size — the fixed
per-call overhead is amortised and the matrix multiplications get better shapes.
This is why `POST /api/feedback/batch` and the CSV importer batch their work
instead of looping over single predictions.

---
## 9. From predictions to business insight

A sentiment label on its own is not actionable. *"23% of feedback is negative"*
prompts the immediate question **"about what?"** — and that is the question the
dashboard has to answer.

### Issue categories

`ISSUE_TAXONOMY` maps keywords to business areas (Delivery & Logistics, Product
Quality, Customer Support, Pricing & Value, …). It is deliberately a transparent
lookup rather than a learned topic model:

- An operations lead can **read** it and **extend** it without retraining
  anything.
- Its output is stable, so week-on-week comparisons are meaningful — an LDA topic
  model re-fit on new data can silently renumber its topics.
- One document can carry several tags, which is correct: a review complaining
  about both delivery *and* support belongs in both queues.

In [ ]:
from src.preprocessing import ISSUE_TAXONOMY, detect_issue_categories

sample = test_df.sample(1200, random_state=RANDOM_STATE).copy()
sample["categories"] = [detect_issue_categories(t) for t in sample.text]

exploded = sample.explode("categories").dropna(subset=["categories"])
issues = (
    exploded.groupby("categories")
    .agg(total=("label_name", "size"),
         negative=("label_name", lambda s: int((s == "negative").sum())))
    .assign(negative_rate=lambda d: (d.negative / d.total * 100).round(1))
    .sort_values("negative", ascending=False)
)
display(issues)

print(f"coverage: {sample.categories.str.len().gt(0).mean():.1%} of feedback "
      f"received at least one category tag")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
ordered = issues.sort_values("negative")
ax.barh(ordered.index, ordered.negative, color="#e34948", height=0.68)
for i, (count, rate) in enumerate(zip(ordered.negative, ordered.negative_rate)):
    ax.text(count + max(ordered.negative) * 0.015, i, f"{rate:.0f}% neg",
            va="center", fontsize=8.5, color="#52514e")
ax.set(xlabel="negative mentions", title="Issue categories ranked by negative volume")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

Ranking by **absolute negative volume** rather than by negative *rate* is a
deliberate choice. A category that is 100% negative across two mentions is noise;
one that is 55% negative across 400 mentions is a real operational problem. Rate
is shown alongside as context, never as the sort key.

### Net Sentiment Score

The dashboard's headline metric is **NSS = %positive − %negative**, ranging from
−100 to +100. It is the standard CX formulation, and it has one property that
matters for a trend chart: it is insensitive to the neutral mass. A week where
lots of indifferent feedback arrives does not move NSS, while a week where
complaints replace praise moves it sharply — which is exactly the signal an
operations team should be alerted on.

In [ ]:
distribution = sample.label_name.value_counts(normalize=True) * 100
nss = distribution.get("positive", 0) - distribution.get("negative", 0)

print(f"positive {distribution.get('positive', 0):.1f}%  "
      f"neutral {distribution.get('neutral', 0):.1f}%  "
      f"negative {distribution.get('negative', 0):.1f}%")
print(f"Net Sentiment Score: {nss:+.1f}")

band = ("Excellent" if nss >= 40 else "Healthy" if nss >= 10 else
        "Mixed" if nss >= -10 else "Poor" if nss >= -40 else "Critical")
print(f"band: {band}")

---
## 10. Conclusions

### What was built

An end-to-end sentiment intelligence system:

- **Preprocessing** — two purpose-built cleaning profiles plus a separate
  topic-extraction path, no external NLP downloads required.
- **Models** — a TF-IDF + Logistic Regression baseline and a fine-tuned
  DistilBERT trained with a hand-written PyTorch loop.
- **Serving** — FastAPI with graceful degradation across four model backends,
  batched inference, and word-level explanations.
- **Storage** — MongoDB Atlas, with all analytics computed by aggregation
  pipelines rather than in application code.
- **Dashboard** — a React SPA with trend, composition, channel, issue and model
  views, plus a live SSE feed.

### What the numbers say

Run the cell below for the final figures from the artefacts in `reports/`.

In [ ]:
delta = bert_metrics['f1_macro'] - baseline_metrics['f1_macro']
best = max(LABELS, key=lambda name: bert_metrics['per_class'][name]['f1'])
worst = min(LABELS, key=lambda name: bert_metrics['per_class'][name]['f1'])

print('FINAL RESULTS (held-out test set, n=%d)' % bert_metrics['n_samples'])
print('=' * 58)
print('  TF-IDF + Logistic Regression   macro-F1  %.4f' % baseline_metrics['f1_macro'])
print('  DistilBERT (fine-tuned)        macro-F1  %.4f' % bert_metrics['f1_macro'])
print('  ' + '-' * 56)
print('  Improvement                              %+.4f  (%+.1f%%)'
      % (delta, delta / baseline_metrics['f1_macro'] * 100))
print()
print('  Best class   %-9s F1 %.3f' % (best, bert_metrics['per_class'][best]['f1']))
print('  Worst class  %-9s F1 %.3f' % (worst, bert_metrics['per_class'][worst]['f1']))
print('  Latency      %s ms per sample on %s'
      % (bert_metrics.get('latency_ms_per_sample', 'n/a'), bert_metrics.get('device', 'cpu')))
print('  Training     %.0f minutes, best epoch %s'
      % (bert_metrics.get('train_seconds', 0) / 60, bert_metrics.get('best_epoch', '?')))

### Limitations — stated plainly

1. **Label noise at the 3-star boundary.** The neutral class is defined by a
   star-rating mapping, and 3-star reviews genuinely mix praise and complaint.
   This puts a practical ceiling on neutral-class performance that no amount of
   modelling removes; better labels would require human annotation.

2. **Domain.** Trained on English business reviews. Short-form social posts,
   code-mixed text (Hinglish and similar) and highly technical support tickets
   are out of distribution, and performance there is unmeasured.

3. **Sarcasm and negation at distance.** *"Great, another week without my
   order"* is the known hard case for every model of this class, and nothing here
   solves it.

4. **A trained artefact reflects its training window.** Language and product
   concerns drift. Without periodic re-evaluation on fresh labelled samples, a
   model quietly degrades — the confidence gap in Section 6 is the cheapest early
   warning signal available.

5. **Scale.** Benchmarks here are CPU-only at moderate volume. High-throughput
   deployment would want GPU serving or ONNX/quantised export.

### Where this should and should not be used

This system belongs **in front of a human triage workflow** — routing, ranking
and summarising, so a team reads the right 5% of feedback first. It should not
take automated action on an individual customer, and low-confidence predictions
should always be reviewed by a person.

### Next steps

- Aspect-based sentiment (per-entity polarity within one review, not per-document)
- Active learning: route the lowest-confidence predictions to human labelling and
  fold the results into the next training round
- Multilingual support via XLM-RoBERTa for non-English markets
- Automated alerting when a category's negative rate breaks its trailing baseline
- ONNX Runtime export and INT8 quantisation for cheaper serving

---

*PulseAI — AI Major Capstone Project · LaunchED Global*